# Campus Chat: From Training Data to Production-Style Serving

**AAIN-7420 Data Engineering for AI — Chapter 7 Complete Build-Together Project**

This notebook is for students who have never built an AI application. It teaches the complete lifecycle:

1. load and validate data;
2. train an intent classifier;
3. build a governed retrieval index;
4. evaluate both components;
5. create registries and a release gate;
6. create a deployment manifest;
7. run local inference;
8. create an evidence trace; and
9. understand how the artifacts become a FastAPI + Streamlit application.

No API key or GPU is required.


## Architecture

The application uses a small trained ML model and a retrieval system.

- **Intent model:** routes a question to registration, advising, financial aid, degree requirements, campus services, technical support, emergency, or greeting.
- **Retriever:** searches only approved, versioned campus documents.
- **Answer composer:** returns the best grounded passage with source metadata.
- **MLOps artifacts:** training run, model registry, prompt registry, corpus manifest, index manifest, benchmark results, release gate, deployment manifest, and evidence trace.


In [ ]:
!pip -q install pandas numpy scipy scikit-learn joblib

In [ ]:
from pathlib import Path
import hashlib, json, uuid
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 120)

BASE = Path("/content/campus_chat") if Path("/content").exists() else Path("campus_chat")
DATA = BASE / "data"
ARTIFACTS = BASE / "artifacts"
EVIDENCE = BASE / "evidence"
for folder in [DATA, ARTIFACTS, EVIDENCE]:
    folder.mkdir(parents=True, exist_ok=True)
print(BASE.resolve())

## 1. Upload the three CSV files

Upload:

- `campus_intents.csv`
- `campus_documents.csv`
- `campus_benchmark.csv`

They are included in the downloadable project package.


In [ ]:
required_files = ["campus_intents.csv", "campus_documents.csv", "campus_benchmark.csv"]
missing = [name for name in required_files if not (DATA / name).exists()]

if missing:
    try:
        from google.colab import files
        print("Upload:", missing)
        uploaded = files.upload()
        for name, content in uploaded.items():
            (DATA / name).write_bytes(content)
    except ImportError:
        print("Place the CSV files in", DATA.resolve())

for name in required_files:
    if not (DATA / name).exists():
        raise FileNotFoundError(f"Missing {name}")

intent_df = pd.read_csv(DATA / "campus_intents.csv")
doc_df = pd.read_csv(DATA / "campus_documents.csv").fillna("")
benchmark_df = pd.read_csv(DATA / "campus_benchmark.csv")
print("Intent examples:", len(intent_df))
print("Documents:", len(doc_df))
print("Benchmark cases:", len(benchmark_df))

## 2. Inspect the training labels

The classifier learns from labeled examples. A label is not just a string; it is a governed model-facing asset. Changing label meaning changes what the model learns.


In [ ]:
display(intent_df.head())
display(intent_df["intent"].value_counts().rename_axis("intent").reset_index(name="examples"))

## 3. Validate and quarantine campus documents

The retriever must not index every incoming record. We require:

- document ID;
- title and text;
- owner;
- effective date;
- source URL;
- document version; and
- public access classification.

Failed records are quarantined instead of silently dropped.


In [ ]:
REQUIRED_FIELDS = ["document_id", "title", "document_text", "owner", "effective_date", "source_url", "document_version"]

def validate_document(row):
    reasons = []
    for field in REQUIRED_FIELDS:
        if not str(row[field]).strip():
            reasons.append(f"missing_{field}")
    if str(row["access_classification"]).lower() != "public":
        reasons.append("non_public_access")
    try:
        pd.to_datetime(row["effective_date"])
    except Exception:
        reasons.append("invalid_effective_date")
    if len(str(row["document_text"]).split()) < 8:
        reasons.append("text_too_short")
    return sorted(set(reasons))

validated = doc_df.copy()
validated["validation_reasons"] = validated.apply(validate_document, axis=1)
validated["validation_status"] = validated["validation_reasons"].map(lambda x: "approved" if not x else "quarantined")
validated["quarantine_reason"] = validated["validation_reasons"].map(lambda x: ";".join(x))
approved_docs = validated.query("validation_status == 'approved'").copy()
quarantine = validated.query("validation_status == 'quarantined'").copy()

print("Approved:", len(approved_docs))
print("Quarantined:", len(quarantine))
display(quarantine[["document_id", "title", "quarantine_reason"]])
quarantine.to_csv(ARTIFACTS / "quarantine_report.csv", index=False)

## 4. Create a corpus snapshot

A corpus snapshot identifies the exact approved document set used for retrieval. It becomes the authoritative release boundary for the index.


In [ ]:
approved_docs["content_hash"] = approved_docs.apply(
    lambda r: hashlib.sha256(
        f"{r['document_id']}||{r['document_version']}||{r['document_text']}".encode("utf-8")
    ).hexdigest(), axis=1
)
corpus_snapshot_id = "campus-corpus-" + hashlib.sha256(
    "||".join(sorted(approved_docs["content_hash"])).encode("utf-8")
).hexdigest()[:12]

corpus_manifest = {
    "corpus_snapshot_id": corpus_snapshot_id,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "document_count": int(len(approved_docs)),
    "documents": approved_docs[["document_id", "document_version", "content_hash"]].to_dict("records")
}
(ARTIFACTS / "corpus_manifest.json").write_text(json.dumps(corpus_manifest, indent=2))
corpus_manifest

## 5. Train the intent classifier

We use a standard, interpretable baseline:

- TF-IDF text features;
- Logistic Regression classifier;
- stratified train/test split;
- fixed random state for reproducibility.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    intent_df["text"], intent_df["intent"],
    test_size=0.25, random_state=42, stratify=intent_df["intent"]
)

intent_vectorizer = TfidfVectorizer(
    lowercase=True, stop_words="english", ngram_range=(1, 2), max_features=5000, sublinear_tf=True
)
X_train_vec = intent_vectorizer.fit_transform(X_train)
X_test_vec = intent_vectorizer.transform(X_test)

intent_model = LogisticRegression(
    max_iter=1500, class_weight="balanced", random_state=42, C=4.0
)
intent_model.fit(X_train_vec, y_train)

predictions = intent_model.predict(X_test_vec)
accuracy = accuracy_score(y_test, predictions)
macro_f1 = f1_score(y_test, predictions, average="macro")
print("Accuracy:", round(accuracy, 3))
print("Macro F1:", round(macro_f1, 3))
print(classification_report(y_test, predictions, zero_division=0))

## 6. Save a reproducible training-run record

A model file is not enough. The run record names the data, parameters, split, code version, and metrics.


In [ ]:
joblib.dump(intent_vectorizer, ARTIFACTS / "intent_vectorizer.joblib")
joblib.dump(intent_model, ARTIFACTS / "intent_model.joblib")

training_run_id = "train-" + uuid.uuid4().hex[:12]
training_run = {
    "training_run_id": training_run_id,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "data_file": "campus_intents.csv",
    "data_snapshot": hashlib.sha256(intent_df.to_csv(index=False).encode("utf-8")).hexdigest()[:12],
    "algorithm": "TF-IDF + LogisticRegression",
    "random_state": 42,
    "test_size": 0.25,
    "metrics": {"accuracy": float(accuracy), "macro_f1": float(macro_f1)},
    "code_version": "campus-chat-notebook-v1"
}
(ARTIFACTS / "training_run.json").write_text(json.dumps(training_run, indent=2))
training_run

## 7. Chunk the approved corpus

Chunking is transformation logic. The chunking version must be tied to the index and deployment manifest.


In [ ]:
CHUNK_SIZE = 65
CHUNK_OVERLAP = 15
CHUNKING_VERSION = "word-window-v1"

def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = str(text).split()
    step = max(1, size - overlap)
    output = []
    for start in range(0, len(words), step):
        chunk = words[start:start + size]
        if not chunk:
            break
        output.append(" ".join(chunk))
        if start + size >= len(words):
            break
    return output

rows = []
for _, row in approved_docs.iterrows():
    for position, chunk in enumerate(chunk_text(row["document_text"]), start=1):
        rows.append({
            "chunk_id": f"{row['document_id']}:{row['document_version']}:c{position:03d}",
            "document_id": row["document_id"],
            "document_version": row["document_version"],
            "title": row["title"],
            "intent": row["intent"],
            "owner": row["owner"],
            "effective_date": row["effective_date"],
            "source_url": row["source_url"],
            "chunk_text": chunk,
            "corpus_snapshot_id": corpus_snapshot_id,
            "chunking_version": CHUNKING_VERSION,
        })
chunks = pd.DataFrame(rows)
chunks.to_csv(ARTIFACTS / "retrieval_chunks.csv", index=False)
print("Chunks:", len(chunks))
display(chunks.head())

## 8. Build the retrieval index

TF-IDF is used so the entire system remains free and local. In a later extension, students can replace it with sentence embeddings while preserving the same lineage fields.


In [ ]:
RETRIEVAL_MODEL_VERSION = "tfidf-retriever-v1"
retrieval_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
retrieval_matrix = retrieval_vectorizer.fit_transform(chunks["chunk_text"])
joblib.dump(retrieval_vectorizer, ARTIFACTS / "retrieval_vectorizer.joblib")
sparse.save_npz(ARTIFACTS / "retrieval_matrix.npz", retrieval_matrix)

index_build_id = "campus-index-" + hashlib.sha256(
    f"{corpus_snapshot_id}|{CHUNKING_VERSION}|{RETRIEVAL_MODEL_VERSION}".encode("utf-8")
).hexdigest()[:12]

index_manifest = {
    "index_build_id": index_build_id,
    "corpus_snapshot_id": corpus_snapshot_id,
    "chunking_version": CHUNKING_VERSION,
    "retrieval_model_version": RETRIEVAL_MODEL_VERSION,
    "chunk_count": int(len(chunks)),
    "built_at_utc": datetime.now(timezone.utc).isoformat(),
}
(ARTIFACTS / "index_manifest.json").write_text(json.dumps(index_manifest, indent=2))
index_manifest

## 9. Evaluate retrieval against a benchmark

The benchmark has an expected intent and expected source document. We calculate hit@1 and hit@3.


In [ ]:
benchmark_rows = []
hit1 = hit3 = 0
for _, row in benchmark_df.iterrows():
    query_vector = retrieval_vectorizer.transform([row["question"]])
    scores = cosine_similarity(query_vector, retrieval_matrix).flatten()
    order = scores.argsort()[::-1][:3]
    retrieved_docs = chunks.iloc[order]["document_id"].tolist()
    row_hit1 = row["expected_document_id"] in retrieved_docs[:1]
    row_hit3 = row["expected_document_id"] in retrieved_docs[:3]
    hit1 += int(row_hit1)
    hit3 += int(row_hit3)
    benchmark_rows.append({
        "benchmark_id": row["benchmark_id"],
        "question": row["question"],
        "expected_document_id": row["expected_document_id"],
        "retrieved_document_ids": "|".join(retrieved_docs),
        "hit_at_1": row_hit1,
        "hit_at_3": row_hit3,
    })
benchmark_results = pd.DataFrame(benchmark_rows)
retrieval_hit_at_1 = hit1 / len(benchmark_df)
retrieval_hit_at_3 = hit3 / len(benchmark_df)
print("Retrieval hit@1:", round(retrieval_hit_at_1, 3))
print("Retrieval hit@3:", round(retrieval_hit_at_3, 3))
display(benchmark_results)
benchmark_results.to_csv(ARTIFACTS / "benchmark_results.csv", index=False)

## 10. Create the model and prompt registries

Registries record reusable artifacts, ownership, evaluation, stage, and rollback information.


In [ ]:
MODEL_VERSION = "intent-classifier-v1"
PROMPT_VERSION = "campus-answer-template-v1"

model_registry = {
    "model_name": "campus_intent_classifier",
    "versions": [{
        "model_version": MODEL_VERSION,
        "training_run_id": training_run_id,
        "metrics": {"accuracy": float(accuracy), "macro_f1": float(macro_f1)},
        "stage": "production-candidate",
        "owner": "Campus Chat ML Team",
        "rollback_version": None,
    }]
}

prompt_registry = {
    "prompt_name": "grounded_campus_answer",
    "versions": [{
        "prompt_version": PROMPT_VERSION,
        "owner": "Campus Chat Product Team",
        "template": "Use only approved retrieved campus content. Cite the source and responsible office.",
        "status": "approved",
        "rollback_version": None,
    }]
}

(ARTIFACTS / "model_registry.json").write_text(json.dumps(model_registry, indent=2))
(ARTIFACTS / "prompt_registry.json").write_text(json.dumps(prompt_registry, indent=2))
model_registry, prompt_registry

## 11. Build a release gate

A release should not be approved from one metric. We require acceptable classifier quality, retrieval quality, and zero critical validation failures.


In [ ]:
checks = {
    "intent_accuracy": {"value": float(accuracy), "threshold": 0.80, "passed": accuracy >= 0.80},
    "intent_macro_f1": {"value": float(macro_f1), "threshold": 0.78, "passed": macro_f1 >= 0.78},
    "retrieval_hit_at_3": {"value": float(retrieval_hit_at_3), "threshold": 0.85, "passed": retrieval_hit_at_3 >= 0.85},
    "critical_validation_failures": {"value": 0, "threshold": 0, "passed": True},
}
release_status = "approved" if all(item["passed"] for item in checks.values()) else "blocked"
release_gate = {
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": release_status,
    "checks": checks,
    "decision_owner": "Campus Chat Release Owner",
}
(ARTIFACTS / "release_gate.json").write_text(json.dumps(release_gate, indent=2))
release_gate

## 12. Create the deployment manifest

The manifest binds the application state that must be promoted and rolled back together.


In [ ]:
deployment_manifest = {
    "application": "campus_chat",
    "deployment_version": "campus-chat-2026.07-v1",
    "release_status": release_status,
    "intent_model_version": MODEL_VERSION,
    "training_run_id": training_run_id,
    "prompt_template_version": PROMPT_VERSION,
    "corpus_snapshot_id": corpus_snapshot_id,
    "chunking_strategy_version": CHUNKING_VERSION,
    "retrieval_model_version": RETRIEVAL_MODEL_VERSION,
    "vector_index_version": index_build_id,
    "benchmark_version": "campus-benchmark-v1",
    "safety_policy_version": "campus-safety-v1",
    "rollout_pattern": "local-demo-then-canary",
    "rollback_target": "none-first-release",
    "owner": "Campus Chat Release Owner",
}
(ARTIFACTS / "deployment_manifest.json").write_text(json.dumps(deployment_manifest, indent=2))
deployment_manifest

## 13. Build the chat inference function

The function classifies intent, retrieves approved chunks, composes a grounded answer, and creates evidence.


In [ ]:
def classify_intent(message):
    vector = intent_vectorizer.transform([message])
    probabilities = intent_model.predict_proba(vector)[0]
    idx = int(probabilities.argmax())
    return intent_model.classes_[idx], float(probabilities[idx])

def retrieve(message, intent, top_k=3):
    query_vector = retrieval_vectorizer.transform([message])
    scores = cosine_similarity(query_vector, retrieval_matrix).flatten()
    order = scores.argsort()[::-1]
    results = []
    for position in order:
        row = chunks.iloc[int(position)]
        if row["intent"] != intent and len(results) < 1:
            continue
        results.append({
            "chunk_id": row["chunk_id"],
            "document_id": row["document_id"],
            "document_version": row["document_version"],
            "title": row["title"],
            "owner": row["owner"],
            "effective_date": row["effective_date"],
            "source_url": row["source_url"],
            "chunk_text": row["chunk_text"],
            "score": round(float(scores[int(position)]), 4),
        })
        if len(results) >= top_k:
            break
    return results

def campus_chat(message):
    request_id = "req-" + uuid.uuid4().hex[:12]
    response_id = "resp-" + uuid.uuid4().hex[:12]
    intent, confidence = classify_intent(message)
    sources = retrieve(message, intent)

    if intent == "emergency":
        answer = "If there is an immediate threat to life or safety, contact local emergency services now and then campus safety when it is safe."
    elif intent == "greeting":
        answer = "Hello! Ask me about registration, advising, degree requirements, financial aid, campus services, or IT support."
    elif confidence < 0.35 or not sources:
        answer = "I am not confident enough to answer from the approved corpus. Please rephrase or contact the responsible office."
    else:
        source = sources[0]
        answer = (
            f"According to {source['title']} (version {source['document_version']}, effective {source['effective_date']}), "
            f"{source['chunk_text']} Source: {source['source_url']}. "
            f"For case-specific decisions, contact {source['owner']}."
        )

    trace = {
        "request_id": request_id,
        "response_id": response_id,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "user_message": message,
        "predicted_intent": intent,
        "intent_confidence": round(confidence, 4),
        "deployment_version": deployment_manifest["deployment_version"],
        "intent_model_version": MODEL_VERSION,
        "prompt_template_version": PROMPT_VERSION,
        "corpus_snapshot_id": corpus_snapshot_id,
        "vector_index_version": index_build_id,
        "retrieved_chunks": [
            {"chunk_id": s["chunk_id"], "document_id": s["document_id"], "score": s["score"]}
            for s in sources
        ],
        "final_response": answer,
        "retention_class": "redacted_evidence_30_days",
    }
    (EVIDENCE / f"{request_id}.json").write_text(json.dumps(trace, indent=2))
    return answer, trace

answer, trace = campus_chat("When is the add drop deadline?")
print(answer)
trace

## 14. Application-level test questions

Try questions from several intents. Inspect the predicted intent, source, confidence, and trace.


In [ ]:
questions = [
    "How do I reset my password?",
    "What happens to financial aid if I drop a course?",
    "How do I schedule advising?",
    "How long can I borrow a library book?",
    "What should I do if I am in immediate danger?",
]
for question in questions:
    answer, trace = campus_chat(question)
    print("
QUESTION:", question)
    print("INTENT:", trace["predicted_intent"], "CONFIDENCE:", trace["intent_confidence"])
    print("ANSWER:", answer[:350], "...")

## 15. From notebook to production-style application

The downloadable project includes:

- `app/api.py`: FastAPI service with `/health`, `/manifest`, and `/chat`.
- `app/streamlit_app.py`: browser-based chat interface.
- `docker-compose.yml`: runs API and UI together.
- `.github/workflows/ci.yml`: rebuilds and tests after code changes.
- `tests/test_pipeline.py`: validation and inference tests.

Local commands:

```bash
python scripts/train_all.py
uvicorn app.api:app --reload --port 8000
streamlit run app/streamlit_app.py
```


## 16. Quiet-failure lab

### Stale corpus
Change a source policy but do not rebuild the index. Which manifest fields become inconsistent?

### Partial rollback
Imagine rolling back the intent model but leaving the prompt and index unchanged. Why can the bad behavior continue?

### Benchmark drift
Change an expected document in the benchmark without creating a new benchmark version. Why can release comparisons become misleading?

### Feedback contamination
Why should a thumbs-down rating not automatically become a training label?


## 17. Final-project artifact checklist

A team can adapt this project and submit:

- working notebook or service;
- synthetic/public dataset;
- validation and quarantine output;
- trained or configured AI component;
- benchmark results;
- training-run record;
- model/prompt/corpus/index registry entries;
- deployment manifest;
- release gate;
- evidence trace;
- architecture diagram;
- README and requirements file.
